# 05. 모델링

`04_eda`까지 생성한 파생변수 포함 데이터셋을 사용해 실거래가 예측 모델을 학습한다.

진행 순서:

1. 최종 파생변수 데이터셋 로드
2. 모델 학습용 컬럼 정리 및 결측 처리
3. 거래일 기준 시간 분할
4. 기준 모델 학습 및 성능 비교
5. 변수 중요도 저장

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(PROJECT_ROOT / '.cache'))
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 1))
(PROJECT_ROOT / '.matplotlib').mkdir(exist_ok=True)
(PROJECT_ROOT / '.cache').mkdir(exist_ok=True)

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = PROJECT_ROOT / 'data/processed/seoul_apt_trade_2025_features.csv'
MODELING_DATA_PATH = PROJECT_ROOT / 'data/processed/modeling_dataset.csv'
SCORE_PATH = PROJECT_ROOT / 'reports/model_scores.csv'
FIGURE_DIR = PROJECT_ROOT / 'reports/figures'
IMPORTANCE_PATH = FIGURE_DIR / 'model_feature_importance.png'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)

## 1. 최종 파생변수 데이터 로드

모델링은 `03_feature_engineering`에서 만든 최종 데이터셋을 기준으로 한다.

In [ ]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
print(f'rows: {len(df):,}')
print(f'columns: {len(df.columns):,}')
df.head()

## 2. 모델 학습용 데이터셋 구성

좌표 생성, 주소 확인, 사후 검증용 컬럼은 모델 입력에서 제외한다. 타깃은 실거래가 총액인 `price_10k_krw`를 사용한다.

`price_per_m2_10k_krw`는 타깃인 거래가를 면적으로 나눈 값이라 누수 가능성이 있으므로 입력 변수에서 제외한다.

In [ ]:
target_col = 'price_10k_krw'

numeric_features = [
    'area_m2',
    'floor',
    'built_year',
    'age',
    'contract_year',
    'contract_month',
    'contract_day',
    'distance_to_cbd_km',
    'distance_to_ybd_km',
    'distance_to_gbd_km',
    'nearest_business_district_distance_km',
    'nearest_subway_distance_km',
    'hospital_count_within_1km',
    'nearest_hospital_distance_km',
    'large_mart_count_within_1km',
]

categorical_features = [
    'gu',
    'law_dong',
    'apartment_name',
    'nearest_business_district',
]

required_columns = [target_col, 'contract_date', *numeric_features, *categorical_features]
modeling_df = df[required_columns].copy()
modeling_df['contract_date'] = pd.to_datetime(modeling_df['contract_date'])

missing_before = modeling_df.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

In [ ]:
rows_before = len(modeling_df)
modeling_df = modeling_df.dropna(subset=[target_col, *numeric_features, *categorical_features]).copy()
rows_after = len(modeling_df)

modeling_df.to_csv(MODELING_DATA_PATH, index=False, encoding='utf-8-sig')

print(f'모델링 데이터 저장: {MODELING_DATA_PATH}')
print(f'제거된 행: {rows_before - rows_after:,}')
print(f'최종 행 수: {rows_after:,}')
modeling_df.head()

## 3. 학습/테스트 데이터 분할

실제 예측 상황과 비슷하게 거래일 기준 시간 분할을 사용한다. 2025년 1월부터 10월까지는 학습, 2025년 11월부터 12월까지는 테스트로 둔다.

In [ ]:
split_date = pd.Timestamp('2025-11-01')
train_df = modeling_df[modeling_df['contract_date'] < split_date].copy()
test_df = modeling_df[modeling_df['contract_date'] >= split_date].copy()

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target_col]
X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target_col]

split_summary = pd.DataFrame({
    'dataset': ['train', 'test'],
    'start_date': [train_df['contract_date'].min(), test_df['contract_date'].min()],
    'end_date': [train_df['contract_date'].max(), test_df['contract_date'].max()],
    'rows': [len(train_df), len(test_df)],
    'target_mean': [y_train.mean(), y_test.mean()],
    'target_median': [y_train.median(), y_test.median()],
})
split_summary

## 4. 모델 파이프라인 정의

설치된 기본 라이브러리만 사용하기 위해 `scikit-learn` 모델로 기준선을 만든다.

- `DummyRegressor`: 평균값 예측 기준선
- `LinearRegression`: 선형회귀
- `RandomForestRegressor`: 랜덤 포레스트
- `ExtraTreesRegressor`: 엑스트라 트리
- `GradientBoostingRegressor`: 그래디언트 부스팅
- `HistGradientBoostingRegressor`: 히스토그램 기반 그래디언트 부스팅

In [ ]:
onehot_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), categorical_features),
    ],
    remainder='drop',
)

ordinal_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), categorical_features),
    ],
    remainder='drop',
)

models = {
    'dummy_mean': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', DummyRegressor(strategy='mean')),
    ]),
    'linear_regression': Pipeline([
        ('preprocess', onehot_preprocessor),
        ('model', LinearRegression()),
    ]),
    'random_forest': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=120,
            max_depth=18,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'extra_trees': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=120,
            max_depth=18,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'gradient_boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.06,
            max_depth=4,
            min_samples_leaf=3,
            random_state=42,
        )),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.06,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            random_state=42,
        )),
    ]),
}

## 5. 모델 학습 및 평가

평가 지표는 회귀 문제에서 자주 쓰는 `MAE`, `RMSE`, `R2`를 사용한다. 금액 단위는 원본 타깃과 같은 만 원이다.

In [ ]:
def evaluate_regression(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred),
    }


def fit_and_evaluate_models(X_train, y_train, X_test, y_test, split_strategy):
    scores = []
    fitted = {}

    for model_name, model in models.items():
        print(f'training: {split_strategy} / {model_name}')
        model.fit(X_train, y_train)
        fitted[model_name] = model

        train_pred = model.predict(X_train)
        test_pred = model.predict(X_test)

        for dataset_name, y_true, y_pred in [
            ('train', y_train, train_pred),
            ('test', y_test, test_pred),
        ]:
            scores.append({
                'split_strategy': split_strategy,
                'model': model_name,
                'dataset': dataset_name,
                **evaluate_regression(y_true, y_pred),
            })

    return pd.DataFrame(scores), fitted

score_df, fitted_models = fit_and_evaluate_models(
    X_train,
    y_train,
    X_test,
    y_test,
    split_strategy='time_2025_01_10_train_11_12_test',
)

score_df.to_csv(SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'성능표 저장: {SCORE_PATH}')
score_df.sort_values(['split_strategy', 'dataset', 'rmse'])

## 7. 랜덤 80:20 분할 성능 비교

전체 데이터를 섞은 뒤 80%는 훈련셋, 20%는 테스트셋으로 사용한다. 시간 분할보다 일반적인 교차 검증 상황에 가까운 비교용 결과다.

In [ ]:
X = modeling_df[numeric_features + categorical_features]
y = modeling_df[target_col]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

random_split_summary = pd.DataFrame({
    'dataset': ['train_random', 'test_random'],
    'rows': [len(X_train_random), len(X_test_random)],
    'target_mean': [y_train_random.mean(), y_test_random.mean()],
    'target_median': [y_train_random.median(), y_test_random.median()],
})
random_split_summary

In [ ]:
random_score_df, random_fitted_models = fit_and_evaluate_models(
    X_train_random,
    y_train_random,
    X_test_random,
    y_test_random,
    split_strategy='random_80_20',
)

score_df = pd.concat([score_df, random_score_df], ignore_index=True)
score_df.to_csv(SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'성능표 업데이트: {SCORE_PATH}')
score_df.sort_values(['split_strategy', 'dataset', 'rmse'])

## 8. 분할 방식별 테스트 성능 비교

In [ ]:
test_scores = score_df[score_df['dataset'].eq('test')].sort_values(['split_strategy', 'rmse']).reset_index(drop=True)
test_scores

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = test_scores.sort_values(['split_strategy', 'rmse'], ascending=[True, False]).copy()
plot_df['label'] = plot_df['split_strategy'] + ' / ' + plot_df['model']
ax.barh(plot_df['label'], plot_df['rmse'], color='#4C78A8')
ax.set_xlabel('RMSE (10k KRW)')
ax.set_ylabel('split / model')
ax.set_title('Test RMSE by Split Strategy')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()
plt.close(fig)

## 9. 변수 중요도 확인

트리 모델 중 `RandomForestRegressor`의 변수 중요도를 확인한다. 범주형 변수는 순서형 인코딩 기준으로 변수 단위 중요도를 해석한다.

In [ ]:
rf_model = fitted_models['random_forest'].named_steps['model']
importance_df = pd.DataFrame({
    'feature': numeric_features + categorical_features,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False)

importance_df.to_csv(PROJECT_ROOT / 'reports/model_feature_importance.csv', index=False, encoding='utf-8-sig')
importance_df

In [ ]:
top_importance = importance_df.head(20).sort_values('importance')

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_importance['feature'], top_importance['importance'], color='#59A14F')
ax.set_xlabel('importance')
ax.set_ylabel('feature')
ax.set_title('RandomForest Feature Importance')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
fig.savefig(IMPORTANCE_PATH, dpi=150, bbox_inches='tight')
print(f'변수 중요도 그림 저장: {IMPORTANCE_PATH}')
plt.show()
plt.close(fig)

## 10. 다음 작업

현재 노트북은 기준 모델 성능 비교까지 수행한다. 이후에는 성능이 가장 좋은 모델을 기준으로 변수 조합, 로그 타깃, 하이퍼파라미터 튜닝, 시간 분할 방식 변경을 실험한다.